# Exp6 Checkpoint Analysis

Analyze contacts-and-distances-v1 model on 1QYS and 7BNY:
1. Generate rollouts, evaluate contact precision and distance accuracy
2. Probe logits after `<long-range-contact>` to see residue contact likelihood
3. Probe logits after `<long-range-contact> <p_i>` for specific residues
4. Repeat with prefix contacts to see conditioning effect

In [ ]:
# Config
CHECKPOINT = "/home/ubuntu/llm-protein-experiments/outputs/exp6/checkpoint-18000"
PDB_IDS = ["1QYS"]  # start with just one protein
N_ROLLOUTS = 2  # fewer rollouts for early checkpoint
MAX_NEW_TOKENS = 1500  # shorter generations
DEVICE = "cuda"

In [ ]:
# Setup
import tempfile, math, time, json
import numpy as np
import torch
import torch.nn.functional as F
from pathlib import Path
from collections import Counter, defaultdict
from biotite.database import rcsb
from biotite.structure.io import pdbx
from biotite.structure import filter_amino_acids
from transformers import LlamaForCausalLM, LlamaConfig
from safetensors.torch import load_file
from experiments.exp6_contact_prediction.src.data import AMINO_ACIDS, VALID_ATOMS, CONTACT_TYPE_TOKENS
from experiments.exp6_contact_prediction.src.train import (
    create_tokenizer, parse_generated_statements, ContactStatement, DistanceStatement
)
import matplotlib.pyplot as plt
%matplotlib inline

NONSTANDARD = {"MSE":"MET","CSE":"CYS","SEC":"CYS","HYP":"PRO","TPO":"THR","SEP":"SER","PTR":"TYR"}

tokenizer = create_tokenizer()

# Load model manually to avoid HF hub validator issue with local paths
ckpt_path = Path(CHECKPOINT)
with open(ckpt_path / "config.json") as f:
    config_dict = json.load(f)
config = LlamaConfig(**config_dict)
model = LlamaForCausalLM(config)
state_dict = load_file(str(ckpt_path / "model.safetensors"))
model.load_state_dict(state_dict)
del state_dict
model = model.to(dtype=torch.bfloat16, device=DEVICE).eval()

end_token_id = tokenizer.convert_tokens_to_ids("<end>")
print(f"Model: {sum(p.numel() for p in model.parameters()):,} params")

# Token IDs we'll need
long_range_id = tokenizer.convert_tokens_to_ids("<long-range-contact>")
medium_range_id = tokenizer.convert_tokens_to_ids("<medium-range-contact>")
short_range_id = tokenizer.convert_tokens_to_ids("<short-range-contact>")
pos_ids = {i: tokenizer.convert_tokens_to_ids(f"<p{i}>") for i in range(2701)}
id_to_pos = {v: k for k, v in pos_ids.items()}

In [ ]:
# Parse PDB structures
def parse_pdb(pdb_id):
    path = rcsb.fetch(pdb_id, "cif", tempfile.gettempdir())
    atoms = pdbx.get_structure(pdbx.CIFFile.read(path).block, model=1)
    chain = atoms[(atoms.chain_id==atoms.chain_id[0]) & filter_amino_acids(atoms) & (atoms.element!="H")]
    unique_res = sorted(set(chain.res_id))
    r2p = {rid:i+1 for i,rid in enumerate(unique_res)}
    sequence = [NONSTANDARD.get(str(chain[chain.res_id==rid].res_name[0]),
                str(chain[chain.res_id==rid].res_name[0])) for rid in unique_res]
    seq_len = len(sequence)
    
    # Compute CB/CA positions for each residue
    cb_positions = {}  # pos -> (x, y, z)
    for rid in unique_res:
        pos = r2p[rid]
        res_atoms = chain[chain.res_id == rid]
        aa = sequence[pos - 1]
        target = "CA" if aa == "GLY" else "CB"
        mask = res_atoms.atom_name == target
        if mask.any():
            cb_positions[pos] = res_atoms.coord[mask][0]
        else:
            # Fallback to CA
            mask = res_atoms.atom_name == "CA"
            if mask.any():
                cb_positions[pos] = res_atoms.coord[mask][0]
    
    # CB-CB distance matrix
    cb_dist = np.full((seq_len, seq_len), np.inf)
    for i in range(1, seq_len + 1):
        for j in range(i + 1, seq_len + 1):
            if i in cb_positions and j in cb_positions:
                d = float(np.linalg.norm(cb_positions[i] - cb_positions[j]))
                cb_dist[i-1, j-1] = d
                cb_dist[j-1, i-1] = d
    
    # GT contacts (CB-CB <= 8A, sep >= 6)
    gt_contacts = {"long": set(), "medium": set(), "short": set()}
    for i in range(1, seq_len + 1):
        for j in range(i + 2, seq_len + 1):
            sep = j - i
            if sep < 6:
                continue
            if cb_dist[i-1, j-1] <= 8.0:
                if sep >= 24:
                    gt_contacts["long"].add((i, j))
                elif sep >= 12:
                    gt_contacts["medium"].add((i, j))
                else:
                    gt_contacts["short"].add((i, j))
    
    # Min CB distance to any residue >= 24 apart (for logit analysis)
    min_long_dist = np.full(seq_len, np.inf)
    for i in range(seq_len):
        for j in range(seq_len):
            if abs(i - j) >= 24 and cb_dist[i, j] < min_long_dist[i]:
                min_long_dist[i] = cb_dist[i, j]
    
    prompt = "<contacts-and-distances-v1> <begin_sequence> " + " ".join(f"<{aa}>" for aa in sequence) + " <begin_statements>"
    
    return {
        "pdb_id": pdb_id, "sequence": sequence, "seq_len": seq_len,
        "cb_dist": cb_dist, "cb_positions": cb_positions,
        "gt_contacts": gt_contacts, "min_long_dist": min_long_dist,
        "prompt": prompt,
    }

proteins = {pdb_id: parse_pdb(pdb_id) for pdb_id in PDB_IDS}
for pdb_id, p in proteins.items():
    gt = p["gt_contacts"]
    print(f"{pdb_id}: {p['seq_len']} res, contacts: short={len(gt['short'])} med={len(gt['medium'])} long={len(gt['long'])}")

In [ ]:
# Generate rollouts and evaluate
def generate_rollout(prompt):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=8192)
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=True,
                                 temperature=1.0, top_k=0, pad_token_id=tokenizer.pad_token_id,
                                 eos_token_id=end_token_id)
    gen_ids = outputs[0][inputs["input_ids"].shape[1]:]
    gen_text = tokenizer.decode(gen_ids, skip_special_tokens=False)
    stmts, valid, plddt = parse_generated_statements(gen_text.split())
    return stmts, valid, plddt

for pdb_id, p in proteins.items():
    print(f"\n{'='*60}")
    print(f"{pdb_id}: Generating {N_ROLLOUTS} rollouts...")
    
    all_rollouts = []
    for r in range(N_ROLLOUTS):
        stmts, valid, plddt = generate_rollout(p["prompt"])
        contacts = [s for s in stmts if isinstance(s, ContactStatement)]
        distances = [s for s in stmts if isinstance(s, DistanceStatement)]
        all_rollouts.append((stmts, valid, plddt, contacts, distances))
        print(f"  R{r}: {len(stmts)} stmts ({len(contacts)}c + {len(distances)}d), valid={valid}, plddt={plddt}")
    p["rollouts"] = all_rollouts
    
    # Contact precision
    gt = p["gt_contacts"]
    gt_all = gt["long"] | gt["medium"] | gt["short"]
    
    for ri, (stmts, valid, plddt, contacts, distances) in enumerate(all_rollouts):
        for ctype, gt_set in [("long", gt["long"]), ("medium", gt["medium"]), ("short", gt["short"])]:
            pred = {(min(c.pos1, c.pos2), max(c.pos1, c.pos2))
                    for c in contacts if c.contact_type == f"{ctype}-range-contact"}
            correct = len(pred & gt_set)
            prec = correct / len(pred) if pred else 0
            if ri == 0:
                print(f"  R0 {ctype}: {correct}/{len(pred)} correct = {prec:.0%} precision ({len(gt_set)} GT)")

In [ ]:
# Distance scatter: predicted vs GT (using first rollout)
fig, axes = plt.subplots(1, len(proteins), figsize=(6*len(proteins), 5))
if len(proteins) == 1:
    axes = [axes]

for ax, (pdb_id, p) in zip(axes, proteins.items()):
    stmts, valid, plddt, contacts, distances = p["rollouts"][0]
    cb_dist = p["cb_dist"]
    seq_len = p["seq_len"]
    
    gt_dists = []
    pred_dists = []
    for d in distances:
        p1, p2 = d.pos1, d.pos2
        if 1 <= p1 <= seq_len and 1 <= p2 <= seq_len:
            gt_d = cb_dist[p1-1, p2-1]  # CB-CB distance (not exactly same atoms but approximate)
            try:
                pred_d = float(d.distance_token[1:])
            except ValueError:
                continue
            if gt_d < 100:  # skip inf
                gt_dists.append(gt_d)
                pred_dists.append(pred_d)
    
    ax.scatter(gt_dists, pred_dists, s=3, alpha=0.3)
    ax.plot([0, 35], [0, 35], "--", color="red", alpha=0.5)
    ax.set_xlabel("GT CB-CB distance (A)")
    ax.set_ylabel("Predicted distance (A)")
    ax.set_title(f"{pdb_id}: Predicted vs GT distance\n(n={len(gt_dists)}, note: different atoms)")
    ax.set_xlim(0, 35)
    ax.set_ylim(0, 35)
    
    # Correlation
    if gt_dists:
        corr = np.corrcoef(gt_dists, pred_dists)[0, 1]
        ax.text(0.05, 0.95, f"r={corr:.2f}", transform=ax.transAxes, va="top", fontsize=10)

plt.tight_layout()
plt.show()
print("Note: GT is CB-CB distance, predicted is random atom pair - correlation expected to be moderate")

In [ ]:
# Logit probing: prompt with <long-range-contact> and check which positions are likely
def get_next_token_probs(prompt_text):
    """Get probability distribution over next token."""
    inputs = tokenizer(prompt_text, return_tensors="pt", truncation=True, max_length=8192)
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    with torch.no_grad():
        logits = model(**inputs).logits[0, -1]  # logits for next token
    probs = torch.softmax(logits, dim=0)
    return probs.cpu()

def position_probs(probs, seq_len):
    """Extract probabilities for position tokens <p1> through <pN>."""
    result = np.zeros(seq_len)
    for i in range(1, seq_len + 1):
        tid = pos_ids.get(i)
        if tid is not None:
            result[i - 1] = probs[tid].item()
    return result

fig, axes = plt.subplots(len(proteins), 1, figsize=(14, 5 * len(proteins)))
if len(proteins) == 1:
    axes = [axes]

for ax, (pdb_id, p) in zip(axes, proteins.items()):
    seq_len = p["seq_len"]
    prompt = p["prompt"] + " <long-range-contact>"
    probs = get_next_token_probs(prompt)
    pos_p = position_probs(probs, seq_len)
    
    # Normalize for visualization
    pos_p_norm = pos_p / pos_p.max() if pos_p.max() > 0 else pos_p
    
    # True min distance to long-range partner
    min_ld = p["min_long_dist"]
    min_ld_clipped = np.clip(min_ld, 0, 30)
    min_ld_norm = 1 - min_ld_clipped / 30  # invert: close = high
    
    x = np.arange(1, seq_len + 1)
    ax.bar(x - 0.2, pos_p_norm, width=0.4, color="#5C9BD5", alpha=0.7, label="P(position | long-range-contact)")
    ax.bar(x + 0.2, min_ld_norm, width=0.4, color="#E57373", alpha=0.5, label="1 - min_dist/30 (true proximity)")
    
    # Horizontal line at 8A cutoff equivalent
    cutoff_norm = 1 - 8.0 / 30
    ax.axhline(cutoff_norm, color="red", linestyle="--", alpha=0.4, label="8A cutoff")
    
    ax.set_xlabel("Residue position")
    ax.set_ylabel("Normalized score")
    ax.set_title(f"{pdb_id}: Which residues does the model predict for <long-range-contact>?")
    ax.legend(fontsize=8)
    ax.set_xlim(0, seq_len + 1)

plt.tight_layout()
plt.show()

In [ ]:
# Logit probing: <long-range-contact> <p_i> -> which p_j?
# Pick a few true long-range contact residues
for pdb_id, p in proteins.items():
    seq_len = p["seq_len"]
    gt_long = p["gt_contacts"]["long"]
    cb_dist = p["cb_dist"]
    
    if not gt_long:
        print(f"{pdb_id}: no long-range contacts")
        continue
    
    # Pick 3 residues that are in long-range contacts
    res_in_long = list({c[0] for c in gt_long} | {c[1] for c in gt_long})
    np.random.seed(42)
    probe_residues = sorted(np.random.choice(res_in_long, size=min(3, len(res_in_long)), replace=False))
    
    fig, axes = plt.subplots(len(probe_residues), 1, figsize=(14, 4 * len(probe_residues)))
    if len(probe_residues) == 1:
        axes = [axes]
    
    for ax, res_i in zip(axes, probe_residues):
        prompt = p["prompt"] + f" <long-range-contact> <p{res_i}>"
        probs = get_next_token_probs(prompt)
        pos_p = position_probs(probs, seq_len)
        
        # True CB distances from this residue
        true_dists = cb_dist[res_i - 1, :]
        
        x = np.arange(1, seq_len + 1)
        
        # Only show positions with sep >= 24
        mask = np.abs(x - res_i) >= 24
        pos_p_masked = np.where(mask, pos_p, 0)
        true_dists_masked = np.where(mask, true_dists, np.nan)
        
        ax2 = ax.twinx()
        ax.bar(x, pos_p_masked, color="#5C9BD5", alpha=0.7, label=f"P(p_j | long-range, p{res_i})")
        ax2.scatter(x[mask], true_dists_masked[mask], s=10, color="#E57373", alpha=0.6, label="True CB-CB dist")
        ax2.axhline(8.0, color="red", linestyle="--", alpha=0.4)
        ax2.set_ylabel("CB-CB distance (A)", color="#E57373")
        ax2.set_ylim(0, 40)
        ax2.invert_yaxis()
        
        # Mark true contacts
        true_partners = [c[1] if c[0]==res_i else c[0] for c in gt_long if res_i in c]
        for tp in true_partners:
            ax.axvline(tp, color="green", alpha=0.3, linewidth=2)
        
        ax.set_xlabel("Residue position")
        ax.set_ylabel("Probability", color="#5C9BD5")
        aa_i = p["sequence"][res_i - 1]
        ax.set_title(f"{pdb_id}: P(p_j | <long-range-contact> <p{res_i}>) — {aa_i} at pos {res_i}, "
                     f"{len(true_partners)} true partners (green lines)")
        ax.legend(loc="upper left", fontsize=8)
        ax2.legend(loc="upper right", fontsize=8)
    
    plt.suptitle(f"{pdb_id}: Logit probing for long-range contact partners", fontsize=13)
    plt.tight_layout()
    plt.show()

In [ ]:
# Same analysis for short and medium contacts
for pdb_id, p in proteins.items():
    seq_len = p["seq_len"]
    cb_dist = p["cb_dist"]
    
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    
    for ax, ctype, sep_range, tid in [
        (axes[0], "short", (6, 12), "<short-range-contact>"),
        (axes[1], "medium", (12, 24), "<medium-range-contact>"),
        (axes[2], "long", (24, 9999), "<long-range-contact>"),
    ]:
        prompt = p["prompt"] + f" {tid}"
        probs = get_next_token_probs(prompt)
        pos_p = position_probs(probs, seq_len)
        
        # Which residues actually have contacts in this range?
        gt_set = p["gt_contacts"][ctype]
        in_contact = np.zeros(seq_len)
        for c in gt_set:
            in_contact[c[0] - 1] = 1
            in_contact[c[1] - 1] = 1
        
        x = np.arange(1, seq_len + 1)
        ax.bar(x, pos_p, color="#5C9BD5", alpha=0.7)
        # Highlight residues that are truly in contacts
        for i in range(seq_len):
            if in_contact[i]:
                ax.bar(i + 1, pos_p[i], color="green", alpha=0.5)
        
        ax.set_xlabel("Residue")
        ax.set_ylabel("P(position)")
        ax.set_title(f"{ctype} ({len(gt_set)} GT)")
    
    fig.suptitle(f"{pdb_id}: P(first position | contact type) — green = in true contacts", fontsize=12)
    plt.tight_layout()
    plt.show()

In [ ]:
# Effect of conditioning: prompt with 1-2 true contacts first
for pdb_id, p in proteins.items():
    gt_long = sorted(p["gt_contacts"]["long"])
    if len(gt_long) < 2:
        print(f"{pdb_id}: too few long-range contacts")
        continue
    
    seq_len = p["seq_len"]
    base = p["prompt"]
    
    # Pick first long-range contact as prefix
    c1 = gt_long[0]
    c2 = gt_long[len(gt_long) // 2]  # one from the middle
    
    prompts = [
        ("No prefix", base + " <long-range-contact>"),
        (f"After ({c1[0]},{c1[1]})",
         base + f" <long-range-contact> <p{c1[0]}> <p{c1[1]}> <long-range-contact>"),
        (f"After ({c1[0]},{c1[1]}) + ({c2[0]},{c2[1]})",
         base + f" <long-range-contact> <p{c1[0]}> <p{c1[1]}> <long-range-contact> <p{c2[0]}> <p{c2[1]}> <long-range-contact>"),
    ]
    
    fig, axes = plt.subplots(len(prompts), 1, figsize=(14, 4 * len(prompts)))
    
    for ax, (label, prompt) in zip(axes, prompts):
        probs = get_next_token_probs(prompt)
        pos_p = position_probs(probs, seq_len)
        
        # Color by whether residue is in a long-range contact
        in_contact = np.zeros(seq_len)
        for c in gt_long:
            in_contact[c[0] - 1] = 1
            in_contact[c[1] - 1] = 1
        
        x = np.arange(1, seq_len + 1)
        colors = ["green" if in_contact[i] else "#5C9BD5" for i in range(seq_len)]
        ax.bar(x, pos_p, color=colors, alpha=0.7)
        ax.set_xlabel("Residue")
        ax.set_ylabel("P(position)")
        ax.set_title(f"{label}")
        ax.set_xlim(0, seq_len + 1)
    
    fig.suptitle(f"{pdb_id}: How does conditioning on true contacts change P(next position)?", fontsize=12)
    plt.tight_layout()
    plt.show()